# Simulation configuration examples

Run a small batch of simulation episodes from one parameter configuration, plot normalized welfare over time, and show network snapshots at selected steps. `CONFIG["mode"]` selects one of two sources:

- `existing_run`: choose the stored parameter row with the highest final normalized welfare for one population size and policy composition.
- `manual`: use the fixed task configuration defined in this notebook.

In [ ]:
import copy
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import scienceplots  # noqa: F401
from IPython.display import display
from matplotlib_inline.backend_inline import set_matplotlib_formats

%matplotlib inline
plt.style.use("science")
set_matplotlib_formats("svg")
plt.rcParams.update({
    "font.size": 14,
    "axes.titlesize": 17,
    "axes.labelsize": 15,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
    "legend.fontsize": 12,
    "legend.title_fontsize": 13,
    "figure.titlesize": 19,
})

project_root = next(
    (
        path
        for path in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
        if (path / "pyproject.toml").is_file() and (path / "tasks").is_dir()
    ),
    None,
)
if project_root is None:
    raise RuntimeError("Run this notebook from the network-creation repository.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from network_creation.core.observability import ObservationSampling
from network_creation.core.runtime import create_simulation_runtime
from network_creation.io import read_json
from network_creation.observers.social_welfare_observer import (
    CenteredMeanWelfareObserver,
)
from network_creation.observers.state_history_observer import StateHistoryObserver
from network_creation.plotting.episode_series import plot_episode_series_observer
from network_creation.plotting.graph_history import build_circle_layout
from network_creation.time_normalization import normalized_source_times
from tasks.policies import load_policy_results
from tasks.policies.agent_parameters import build_sampled_agent_parameter_plan
from tasks.policies.agents import PolicyAgentBuilder
from tasks.policies.sweep import (
    ParameterSamples,
    build_sweep,
    collect_parameter_specs,
    parse_policy_entries,
    sample_parameters,
)

In [ ]:
CONFIG = {
    "mode": "existing_run",  # "existing_run" or "manual"
    "existing_run": {
        "result_store": (
            project_root
            / "artifacts"
            / "simulation_runs"
            / "run-20260713T235130.zarr"
        ),
        "n_agents": 100,
        "composition": 1.0,
        "objective": "centered_mean_welfare.mean",
        "normalize_time_for_selection": True,
    },
    "manual": {
        "sampling": {"log2_budget": 0, "seed": 123},
        "simulator": {
            "seed": 19,
            "max_steps": 800,
            "n_episodes": 1,
            "actor_scheduler": {"n_actors": 1},
            "initial_graph": {"kind": "empty"},
        },
        "game": {
            "benefit_to_cost_ratio": {"kind": "fixed", "value": 0.1},
            "distance_discount": {"kind": "fixed", "value": 0.9},
        },
        "population": {
            "n_agents": [30],
            "visibility_distance": {"kind": "fixed", "value": 5},
            "policies": [
                {
                    "id": "best_response",
                    "kind": "best-response",
                    "parameters": {
                        "outside_optimism": {"kind": "fixed", "value": 0.43},
                        "exploration": {"kind": "fixed", "value": 0.08},
                    },
                }
            ],
            "policy_proportions": {
                "policy": "best_response",
                "values": [1.0],
            },
        },
    },
    "rerun": {
        "seed": None,  # None keeps the source seed
        "max_steps": None,  # None keeps the source horizon
        "n_episodes": 6,
        "sample_every": 10,
    },
}

In [ ]:
def composition_dimension(results):
    dimensions = tuple(
        dim
        for dim in results["x"].dims
        if dim not in {"n_agents", "parameter_row", "step", "observable"}
    )
    if len(dimensions) != 1:
        raise ValueError(f"Expected one policy-composition dimension, found {dimensions}.")
    return dimensions[0]


def select_composition(results, dimension, requested):
    values = np.asarray(results.coords[dimension].values, dtype=float)
    matches = np.flatnonzero(np.isclose(values, requested, rtol=0.0, atol=1e-7))
    if matches.size != 1:
        raise ValueError(
            f"Composition {requested} did not uniquely match {values.tolist()}."
        )
    return results.coords[dimension].values[int(matches[0])].item()


def saved_run_config(results, result_store):
    configured_path = Path(results.attrs["config_path"])
    path = configured_path if configured_path.is_file() else result_store / "run_config.json"
    config = read_json(path)
    if not isinstance(config, dict):
        raise ValueError(f"Expected a JSON object in {path}.")
    return config


def narrow_structure(config, n_agents, composition):
    config = copy.deepcopy(config)
    population = config["population"]
    population["n_agents"] = [int(n_agents)]
    if "policy_proportions" in population:
        population["policy_proportions"]["values"] = [float(composition)]
    elif "policy_counts" in population:
        population["policy_counts"]["values"] = [int(composition)]
    else:
        raise ValueError("Population must define policy_proportions or policy_counts.")
    return config


def existing_run_source(options):
    result_store = Path(options["result_store"])
    objective = str(options["objective"])
    results = load_policy_results(
        result_store,
        stats=(objective,),
        normalize_time=bool(options["normalize_time_for_selection"]),
    )
    dimension = composition_dimension(results)
    composition = select_composition(results, dimension, options["composition"])
    final_values = results["x"].sel(
        {
            "observable": objective,
            "n_agents": int(options["n_agents"]),
            dimension: composition,
        }
    ).isel(step=-1, drop=True)
    values = np.asarray(final_values.values, dtype=float)
    finite_positions = np.flatnonzero(np.isfinite(values))
    if not finite_positions.size:
        raise ValueError("No finite welfare values match the requested configuration.")
    best_position = int(finite_positions[np.argmax(values[finite_positions])])
    parameter_row = int(final_values.coords["parameter_row"].values[best_position])
    selector = {
        "n_agents": int(options["n_agents"]),
        dimension: composition,
        "parameter_row": parameter_row,
    }
    theta_row = results["theta"].sel(selector)
    theta = {
        str(name): float(value)
        for name, value in zip(
            theta_row.coords["theta_parameter"].values,
            theta_row.values,
            strict=True,
        )
    }
    run_config = narrow_structure(
        saved_run_config(results, result_store),
        options["n_agents"],
        composition,
    )
    parameter_specs = collect_parameter_specs(run_config)["parameters"]
    parameter_names = tuple(parameter_specs)
    missing = set(parameter_names).difference(theta)
    extra = set(theta).difference({"n_agents", *parameter_names})
    if missing or extra:
        raise ValueError(
            f"Stored parameters do not match the run config: "
            f"missing={sorted(missing)}, extra={sorted(extra)}"
        )
    samples = ParameterSamples(
        parameter_names=parameter_names,
        parameter_values=np.asarray(
            [[theta[name] for name in parameter_names]], dtype=np.float32
        ),
        column_index_by_name={name: index for index, name in enumerate(parameter_names)},
    )
    return {
        "config": run_config,
        "parameter_samples": samples,
        "sample_key": int(results.coords["input_index"].sel(selector).item()),
        "summary": {
            "source": str(result_store.relative_to(project_root)),
            "parameter_row": parameter_row,
            "stored_objective": float(values[best_position]),
            "selection_step": float(results.coords["step"].values[-1]),
            "selection_time_axis": results.attrs.get(
                "time_axis", "raw_simulator_steps"
            ),
            "parameters": theta,
        },
    }


def manual_source(config):
    config = copy.deepcopy(config)
    samples = sample_parameters(collect_parameter_specs(config))
    if samples.parameter_values.shape[0] != 1:
        raise ValueError("Manual mode must resolve to exactly one parameter row.")
    return {
        "config": config,
        "parameter_samples": samples,
        "sample_key": 0,
        "summary": {
            "source": "manual configuration",
            "parameters": {
                name: float(value)
                for name, value in zip(
                    samples.parameter_names,
                    samples.parameter_values[0],
                    strict=True,
                )
            },
        },
    }


def prepare_source(config):
    if config["mode"] == "existing_run":
        return existing_run_source(config["existing_run"])
    if config["mode"] == "manual":
        return manual_source(config["manual"])
    raise ValueError("CONFIG['mode'] must be 'existing_run' or 'manual'.")


def run_source(source, rerun):
    config = copy.deepcopy(source["config"])
    simulator_config = config["simulator"]
    simulator_config["n_episodes"] = int(rerun["n_episodes"])
    if rerun["seed"] is not None:
        simulator_config["seed"] = int(rerun["seed"])
    if rerun["max_steps"] is not None:
        simulator_config["max_steps"] = int(rerun["max_steps"])

    policy_entries = parse_policy_entries(config)
    policy_ids = tuple(policy_id for policy_id, _ in policy_entries)
    parameter_specs = collect_parameter_specs(config)["parameters"]
    sweep = build_sweep(
        config,
        source["parameter_samples"],
        parameter_specs=parameter_specs,
        policy_ids=policy_ids,
    )
    if sweep.total_inputs != 1:
        raise ValueError(f"Expected one rerun input, found {sweep.total_inputs}.")
    configuration = sweep.configuration_at(0)
    settings = sweep.resolve(configuration)
    sampled_plan = build_sampled_agent_parameter_plan(
        parameter_specs,
        configuration_column_by_name=sweep.column_index_by_name,
        policy_entries=policy_entries,
    )
    agent_builder = PolicyAgentBuilder(
        policy_entries=policy_entries,
        configuration_names=sweep.configuration_names,
        column_index_by_name=sweep.column_index_by_name,
        sampled_parameters_by_policy_id=sampled_plan.parameters_by_policy_id,
    )
    sampled_agent_parameters = sampled_plan.sample(
        source["sample_key"],
        configuration,
        settings,
        max(sweep.n_agents_values),
        agent_builder.policy_entry_for_node,
    )
    benefit_to_cost_ratio = float(
        configuration[sweep.column_index_by_name["benefit_to_cost_ratio"]]
    )
    distance_discount = float(
        configuration[sweep.column_index_by_name["distance_discount"]]
    )
    welfare_observer = CenteredMeanWelfareObserver(
        n_agents=settings.n_agents,
        observation_sampling=ObservationSampling.periodic(
            int(rerun["sample_every"])
        ),
        benefit_to_cost_ratio=benefit_to_cost_ratio,
        distance_discount=distance_discount,
        aggregations=("mean", "std", "runs"),
    )
    graph_observer = StateHistoryObserver(
        observation_sampling=ObservationSampling.periodic(
            int(rerun["sample_every"])
        ),
        keep_runs=True,
    )
    _, _, simulator = create_simulation_runtime(
        simulation_settings=settings,
        build_agent=lambda node_id: agent_builder.build_agent(
            node_id,
            configuration,
            settings,
            sampled_agent_parameters=sampled_agent_parameters,
        ),
        observers=(welfare_observer, graph_observer),
        seed=settings.seed,
    )
    simulator.run(n_episodes=settings.n_episodes, reset_observers=True)
    return {
        "settings": settings,
        "configuration": {
            name: float(configuration[index])
            for name, index in sweep.column_index_by_name.items()
        },
        "welfare_observer": welfare_observer,
        "graph_result": graph_observer.get(),
    }


def plot_simulation_summary(run, *, episode, times, graph_side):
    result = run["graph_result"]
    steps = result.runs_steps[episode]
    graphs = result.runs_graphs[episode]
    normalized_times = normalized_source_times(
        steps,
        n_agents=run["settings"].n_agents,
        n_actors=run["settings"].n_actors,
    )
    graph_indices = [
        int(np.abs(normalized_times - time).argmin()) for time in times
    ]
    snapshots = [
        (float(normalized_times[index]), graphs[index]) for index in graph_indices
    ]
    series_height = 3.0
    with plt.ioff():
        figure = plt.figure(
            figsize=(graph_side * len(snapshots), series_height + graph_side + 0.5),
            layout="compressed",
        )
    grid = figure.add_gridspec(
        2,
        len(snapshots),
        height_ratios=(series_height, graph_side),
    )

    welfare_observer = run["welfare_observer"]
    welfare_times = normalized_source_times(
        welfare_observer.steps,
        n_agents=run["settings"].n_agents,
        n_actors=run["settings"].n_actors,
    )
    series_axis = figure.add_subplot(grid[0, :])
    plot_episode_series_observer(
        observer=welfare_observer,
        ax=series_axis,
        show_runs=True,
        x_values=welfare_times,
        x_label="activation opportunities per agent",
        band_color="#1f4e79",
    )
    series_axis.set_title("Normalized welfare across rerun episodes")

    with plt.rc_context():
        for column, (time, graph) in enumerate(snapshots):
            axis = figure.add_subplot(grid[1, column])
            graph_nx = graph.to_networkx(vertex_attr_hashable="name")
            nodes = tuple(graph_nx)
            circle = build_circle_layout(len(nodes))
            positions = {
                node: np.asarray(circle[index], dtype=float)
                for index, node in enumerate(nodes)
            }

            nx.draw_networkx_edges(
                graph_nx,
                pos=positions,
                width=0.45,
                edge_color="#1f4e79",
                alpha=0.28,
                arrows=False,
                ax=axis,
            )
            nx.draw_networkx_nodes(
                graph_nx,
                pos=positions,
                node_size=18,
                node_color="#1f4e79",
                node_shape="o",
                edgecolors="#18354d",
                linewidths=0.35,
                ax=axis,
            )
            axis.set(xlim=(-1.08, 1.08), ylim=(-1.08, 1.08), aspect="equal")
            axis.set_axis_off()
            axis.text(
                0.5,
                -0.04,
                f"Time {time:g}",
                transform=axis.transAxes,
                ha="center",
                va="top",
            )
    return figure

In [ ]:
source = prepare_source(CONFIG)
display(source["summary"])

In [ ]:
run = run_source(source, CONFIG["rerun"])
display({
    "n_agents": run["settings"].n_agents,
    "n_actors": run["settings"].n_actors,
    "max_steps": run["settings"].max_steps,
    "n_episodes": run["welfare_observer"].n_episodes,
    "resolved_configuration": run["configuration"],
})

In [ ]:
graph_episode = 0
graph_times = [0, 10, 20, 30]
graph_side = 2.0

summary_figure = plot_simulation_summary(
    run,
    episode=graph_episode,
    times=graph_times,
    graph_side=graph_side,
)
display(summary_figure)
plt.close(summary_figure)

In [ ]:
run_id = Path(CONFIG["existing_run"]["result_store"]).stem
figure_dir = project_root / "artifacts" / "figures" / run_id
figure_dir.mkdir(parents=True, exist_ok=True)
target_stat = CONFIG["existing_run"]["objective"]
pdf_path = figure_dir / f"{target_stat}__timeseries__n_agents-{run['settings'].n_agents}.pdf"
summary_figure.savefig(pdf_path, format="pdf", bbox_inches="tight")
print(f"Saved {pdf_path.relative_to(project_root)}")